# QDM Phase 2B: Real Quantization + Pruning Differentiation

**What this adds beyond Phase 2A:**
1. **Real bitsandbytes LLM.int8** — applied via HuggingFace, with a calibration check that the SAE still produces sensible features on HF activations
2. **Real bitsandbytes NF4 (4-bit)** — same pipeline
3. **Magnitude pruning at matched perplexity** — sparsity binary-searched to match a quantization condition's perplexity delta
4. **Matched-perplexity comparison metrics:**
   - Jaccard overlap of damaged features between quantization and pruning
   - Per-feature damage-score correlation between methods
5. **Discrepancy plot:** which features die under quantization but survive pruning (and vice versa)

**Prerequisites:** Phase 2A must have completed and saved its CSVs and feature tensors to Drive.

**Compute:** ~30-60 min on A100. Most time is bitsandbytes model loading + perplexity search for pruning.

**Headline question:** At matched perplexity, do quantization and pruning damage *the same features*, or different feature classes? Low overlap = a strong contribution distinct from Borobia et al.

## 1. Drive and config (must match Phase 2A)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/qdm_phase2a_70m_results'  # same dir as Phase 2A
PHASE2B_DIR = '/content/drive/MyDrive/qdm_phase2b_70m_results'
os.makedirs(PHASE2B_DIR, exist_ok=True)

CKPT_2A = lambda name: os.path.join(RESULTS_DIR, name)
CKPT_2B = lambda name: os.path.join(PHASE2B_DIR, name)

# Verify Phase 2A results exist
PRIMARY_LAYER = 4
required = [
    f"phase2a_L{PRIMARY_LAYER}_summary.csv",
    f"phase2a_L{PRIMARY_LAYER}_features_FP16.pt",
    f"phase2a_L{PRIMARY_LAYER}_firing_rate_FP16.pt",
]
for r in required:
    p = CKPT_2A(r)
    assert os.path.exists(p), f"Missing Phase 2A artifact: {p}. Run Phase 2A first."
print("Phase 2A artifacts present.")
print(f"Phase 2B output dir: {PHASE2B_DIR}")

In [ ]:
!pip install -q transformer_lens sae-lens datasets matplotlib pandas bitsandbytes accelerate
print("Installed.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from datasets import load_dataset
from transformer_lens import HookedTransformer
from sae_lens import SAE
from tqdm.auto import tqdm

assert torch.cuda.is_available()
device = "cuda"
torch.set_grad_enabled(False)
print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Config — must match Phase 2A
MODEL_NAME = "pythia-70m-deduped"
HF_MODEL_NAME = "EleutherAI/pythia-70m-deduped"
SAE_RELEASE = "pythia-70m-deduped-res-sm"

LAYER = 4
TOKEN_BUDGET = 200_000
SEQ_LEN = 512
BATCH_SIZE = 16

# Which RTN bit-width to use as the "matched perplexity" target for pruning
PRUNING_MATCH_TARGET = "RTN_INT6"  # change if your Phase 2A INT6 didn't have meaningful damage

print(f"Layer: {LAYER}, tokens: {TOKEN_BUDGET:,}, pruning matched to: {PRUNING_MATCH_TARGET}")

## 2. Load TL model + SAE + tokens (same setup as Phase 2A)

In [ ]:
print("Loading TL model and SAE...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()
original_state = {k: v.clone() for k, v in model.state_dict().items()}

hook_name = f"blocks.{LAYER}.hook_resid_post"
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=hook_name, device=device)
sae.eval()

# Same tokens as Phase 2A
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
full_text = "\n\n".join(x for x in ds["text"] if x.strip())
tokens = model.tokenizer(full_text, return_tensors="pt", truncation=False)["input_ids"][0]
n_seqs = TOKEN_BUDGET // SEQ_LEN
tokens_2d = tokens[:n_seqs * SEQ_LEN].reshape(n_seqs, SEQ_LEN).to(device)
tokens_2d_cpu = tokens_2d.cpu()

print(f"Tokens: {tokens_2d.shape}")

# Load Phase 2A FP16 features
features_fp16 = torch.load(CKPT_2A(f"phase2a_L{LAYER}_features_FP16.pt"))
fp16_firing_rate = torch.load(CKPT_2A(f"phase2a_L{LAYER}_firing_rate_FP16.pt"))
ppl_fp16 = float(np.load(CKPT_2A(f"phase2a_L{LAYER}_ppl_FP16.npy"))[0])
print(f"FP16 baseline ppl: {ppl_fp16:.3f}, active features: {(fp16_firing_rate > 0.001).sum().item()}")

## 3. Helpers (cache, encode, perplexity, metrics)

In [ ]:
WEIGHT_KEYWORDS = ["W_Q", "W_K", "W_V", "W_O", "W_in", "W_out"]

def is_quantizable(name):
    return any(s in name for s in WEIGHT_KEYWORDS)

def restore(model, state):
    model.load_state_dict(state)

def cache_tl_activations(model, tokens_2d, hook_name, batch_size):
    storage = []
    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc="caching", leave=False):
        batch = tokens_2d[i:i+batch_size]
        _, cache = model.run_with_cache(batch, names_filter=[hook_name])
        storage.append(cache[hook_name].cpu())
    acts = torch.cat(storage, dim=0)
    return acts.reshape(-1, acts.shape[-1])

def compute_tl_perplexity(model, tokens_2d, batch_size):
    losses = []
    for i in range(0, tokens_2d.shape[0], batch_size):
        loss = model(tokens_2d[i:i+batch_size], return_type="loss")
        losses.append(loss.item())
    return float(np.exp(float(np.mean(losses))))

def sae_encode_batched(sae, acts, device, batch=8192):
    out = []
    for i in tqdm(range(0, acts.shape[0], batch), desc="SAE encoding", leave=False):
        chunk = acts[i:i+batch].to(device).float()
        out.append(sae.encode(chunk).cpu().to(torch.float16))
    return torch.cat(out, dim=0)

def per_feature_pearson(a, b, eps=1e-8):
    a = a.float()
    b = b.float()
    a_c = a - a.mean(dim=0, keepdim=True)
    b_c = b - b.mean(dim=0, keepdim=True)
    num = (a_c * b_c).sum(dim=0)
    den = torch.sqrt((a_c**2).sum(dim=0) * (b_c**2).sum(dim=0)) + eps
    return num / den

def summarize(corrs, firing, threshold=0.001):
    active = corrs[firing > threshold]
    return {
        "n_active": int((firing > threshold).sum().item()),
        "mean_corr": float(active.mean()),
        "median_corr": float(active.median()),
        "survived_pct": float((active > 0.9).float().mean()) * 100,
        "degraded_pct": float(((active > 0.5) & (active <= 0.9)).float().mean()) * 100,
        "damaged_pct": float((active < 0.5).float().mean()) * 100,
    }

print("Helpers ready.")

## 4. Magnitude pruning + binary search for matched perplexity

In [ ]:
def apply_magnitude_pruning(model, sparsity):
    """Zero the smallest-magnitude (sparsity) fraction of each weight tensor."""
    count = 0
    for name, param in model.named_parameters():
        if not is_quantizable(name):
            continue
        w = param.data
        if sparsity > 0:
            flat_abs = w.abs().flatten()
            k = int(sparsity * flat_abs.numel())
            if k > 0:
                threshold = torch.kthvalue(flat_abs, k).values
                mask = w.abs() > threshold
                param.data = (w * mask).to(w.dtype)
        count += 1
    return count

def find_matched_sparsity(model, tokens_2d, original_state, target_ppl,
                          ppl_tol=0.5, max_iters=8, batch_size=16):
    """Binary-search magnitude-pruning sparsity to match target perplexity."""
    lo, hi = 0.0, 0.85  # very aggressive upper bound
    best = None
    history = []
    for it in range(max_iters):
        mid = (lo + hi) / 2
        restore(model, original_state)
        apply_magnitude_pruning(model, sparsity=mid)
        ppl = compute_tl_perplexity(model, tokens_2d, batch_size)
        history.append((mid, ppl))
        print(f"  iter {it}: sparsity={mid:.3f} → ppl={ppl:.2f} (target {target_ppl:.2f})")
        if abs(ppl - target_ppl) < ppl_tol:
            best = (mid, ppl)
            break
        if best is None or abs(ppl - target_ppl) < abs(best[1] - target_ppl):
            best = (mid, ppl)
        if ppl < target_ppl:
            lo = mid  # need more pruning
        else:
            hi = mid  # too aggressive
    restore(model, original_state)
    return best, history

# Read the Phase 2A summary to find target perplexity
phase2a_summary = pd.read_csv(CKPT_2A(f"phase2a_L{LAYER}_summary.csv"))
target_row = phase2a_summary[phase2a_summary["condition"] == PRUNING_MATCH_TARGET].iloc[0]
target_ppl = float(target_row["perplexity"])
print(f"Matching pruning to {PRUNING_MATCH_TARGET}: target ppl = {target_ppl:.3f}")

In [ ]:
# Run the search
print(f"\nBinary-searching pruning sparsity for ppl ≈ {target_ppl:.2f}...")
best_match, history = find_matched_sparsity(
    model, tokens_2d, original_state, target_ppl=target_ppl,
    ppl_tol=0.5, max_iters=8, batch_size=BATCH_SIZE
)
matched_sparsity, matched_ppl = best_match
print(f"\nBest match: sparsity={matched_sparsity:.3f}, ppl={matched_ppl:.3f}")
print(f"  target was {target_ppl:.3f} (delta {(matched_ppl/target_ppl - 1)*100:+.2f}%)")

## 5. Cache pruning activations and compute features

Run the matched-sparsity pruning, cache activations, compute correlations vs FP16.

In [ ]:
print("Applying matched-sparsity pruning and caching activations...")
restore(model, original_state)
apply_magnitude_pruning(model, sparsity=matched_sparsity)
ppl_prune = compute_tl_perplexity(model, tokens_2d, BATCH_SIZE)

acts_prune = cache_tl_activations(model, tokens_2d, hook_name, BATCH_SIZE)
features_prune = sae_encode_batched(sae, acts_prune, device)
corrs_prune = per_feature_pearson(features_fp16, features_prune)
summary_prune = summarize(corrs_prune, fp16_firing_rate)
restore(model, original_state)

print(f"\nPruning condition (sparsity={matched_sparsity:.3f}):")
print(f"  ppl: {ppl_prune:.3f} (Δ {(ppl_prune/ppl_fp16-1)*100:+.2f}%)")
print(f"  survived: {summary_prune['survived_pct']:.2f}%")
print(f"  degraded: {summary_prune['degraded_pct']:.2f}%")
print(f"  damaged:  {summary_prune['damaged_pct']:.2f}%")

# Save
torch.save(corrs_prune, CKPT_2B(f"phase2b_corrs_PRUNE_match{PRUNING_MATCH_TARGET}.pt"))
prune_row = {
    "condition": f"PRUNE_match_{PRUNING_MATCH_TARGET}",
    "method": "pruning",
    "param": matched_sparsity,
    "perplexity": ppl_prune,
    "ppl_delta_pct": (ppl_prune/ppl_fp16 - 1) * 100,
    **summary_prune
}
print(prune_row)
del acts_prune, features_prune; gc.collect(); torch.cuda.empty_cache()

## 6. Free TL model and load HF model for bitsandbytes

The TL model and HF model can't both fit in memory comfortably on Pythia-70M (they would fit but bitsandbytes' device_map is finicky). Cleanest is to swap.

In [ ]:
# Keep features_fp16 and fp16_firing_rate in CPU RAM; free everything else GPU-related
del model
gc.collect()
torch.cuda.empty_cache()
print("TL model freed.")

## 7. Load HF model FP16 baseline + calibration check

Before trusting bitsandbytes-on-HF features, verify the SAE still produces sensible features when fed HF activations. We do this by:

1. Loading the HF model at FP16
2. Running it on the same tokens, extracting hidden states at the equivalent layer
3. Encoding with the same SAE
4. Computing Pearson correlation between *FP16 HF features* and *FP16 TL features* (from Phase 2A)

If that correlation is high (say > 0.9 for most features), the HF activations are close enough to TL activations that the SAE works on both. Then we can trust the bitsandbytes comparisons.

If it's low, the HF activations differ too much and we need to either (a) accept that we're measuring "bitsandbytes vs TL FP16 reference" which is a noisier comparison, or (b) train a small calibration SAE.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading HF model in FP16: {HF_MODEL_NAME}...")
hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
hf_model.eval()
print("HF FP16 loaded.")

In [ ]:
# Pythia HF: gpt_neox.layers[LAYER]
# We hook the OUTPUT of that layer to get post-residual activations

def cache_hf_activations(hf_model, tokens_2d_cpu, layer_idx, batch_size):
    """Hook the output of gpt_neox.layers[layer_idx] and collect hidden states."""
    target = hf_model.gpt_neox.layers[layer_idx]
    captured = []

    def hook(module, inputs, output):
        # GPTNeoXLayer returns a tuple where output[0] is hidden states
        hs = output[0] if isinstance(output, tuple) else output
        captured.append(hs.detach().cpu().float())

    handle = target.register_forward_hook(hook)
    try:
        for i in tqdm(range(0, tokens_2d_cpu.shape[0], batch_size), desc="HF caching", leave=False):
            batch = tokens_2d_cpu[i:i+batch_size].to(hf_model.device)
            _ = hf_model(batch)
    finally:
        handle.remove()

    acts = torch.cat(captured, dim=0)
    return acts.reshape(-1, acts.shape[-1])

def compute_hf_perplexity(hf_model, tokens_2d_cpu, batch_size):
    losses = []
    for i in range(0, tokens_2d_cpu.shape[0], batch_size):
        batch = tokens_2d_cpu[i:i+batch_size].to(hf_model.device)
        out = hf_model(batch, labels=batch)
        losses.append(out.loss.item())
    return float(np.exp(float(np.mean(losses))))

# Calibration: cache HF FP16 activations
print("Caching HF FP16 activations...")
acts_hf_fp16 = cache_hf_activations(hf_model, tokens_2d_cpu, LAYER, BATCH_SIZE)
ppl_hf_fp16 = compute_hf_perplexity(hf_model, tokens_2d_cpu, BATCH_SIZE)
print(f"HF FP16 ppl: {ppl_hf_fp16:.3f} (TL FP16 was {ppl_fp16:.3f})")
print(f"  if these differ by more than ~5%, HF/TL conventions diverge meaningfully")

In [ ]:
# Encode HF FP16 acts with the same SAE and compare to TL FP16 features
features_hf_fp16 = sae_encode_batched(sae, acts_hf_fp16, device)
corrs_hf_calibration = per_feature_pearson(features_fp16, features_hf_fp16)

# Restrict to active features and report
active_mask = fp16_firing_rate > 0.001
calib_active = corrs_hf_calibration[active_mask]
print(f"\n=== Calibration: TL FP16 vs HF FP16 features ===")
print(f"  Mean active-feature correlation: {calib_active.mean():.3f}")
print(f"  Median: {calib_active.median():.3f}")
print(f"  Survived (>0.9): {(calib_active > 0.9).float().mean()*100:.1f}%")

torch.save(corrs_hf_calibration, CKPT_2B("phase2b_calibration_HF_FP16.pt"))

if calib_active.mean() > 0.85:
    print("\n✓ Calibration good. HF features ≈ TL features. Proceed with bitsandbytes.")
    USE_TL_REFERENCE = True
    hf_fp16_reference_features = features_fp16  # use TL features as reference
elif calib_active.mean() > 0.7:
    print("\n⚠ Calibration moderate. Bitsandbytes results should be interpreted as 'vs HF FP16 reference', not vs TL FP16.")
    print("  Switching to HF FP16 as the bitsandbytes reference.")
    USE_TL_REFERENCE = False
    hf_fp16_reference_features = features_hf_fp16
else:
    print("\n✗ Calibration weak. HF and TL features diverge substantially.")
    print("  Bitsandbytes results will be reported relative to HF FP16 reference, but treat them as approximate.")
    USE_TL_REFERENCE = False
    hf_fp16_reference_features = features_hf_fp16

# Free HF FP16 acts (keep features for reference if needed)
del acts_hf_fp16
gc.collect()
torch.cuda.empty_cache()

## 8. Bitsandbytes LLM.int8 condition

Load Pythia-70M with bitsandbytes 8-bit, cache activations, compute correlations vs the chosen FP16 reference.

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()

print("Loading HF model with bitsandbytes INT8...")
hf_model_int8 = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME, load_in_8bit=True, device_map="auto"
)
hf_model_int8.eval()
print("HF INT8 loaded.")

print("\nCaching INT8 activations...")
acts_int8 = cache_hf_activations(hf_model_int8, tokens_2d_cpu, LAYER, BATCH_SIZE)
ppl_int8 = compute_hf_perplexity(hf_model_int8, tokens_2d_cpu, BATCH_SIZE)

features_int8 = sae_encode_batched(sae, acts_int8, device)
corrs_int8 = per_feature_pearson(hf_fp16_reference_features, features_int8)
summary_int8 = summarize(corrs_int8, fp16_firing_rate)

print(f"\n=== bitsandbytes INT8 ===")
print(f"  ppl: {ppl_int8:.3f} (Δ vs HF FP16: {(ppl_int8/ppl_hf_fp16-1)*100:+.2f}%)")
print(f"  survived: {summary_int8['survived_pct']:.2f}%")
print(f"  damaged: {summary_int8['damaged_pct']:.2f}%")

torch.save(corrs_int8, CKPT_2B("phase2b_corrs_BNB_INT8.pt"))

del hf_model_int8, acts_int8, features_int8
gc.collect()
torch.cuda.empty_cache()

## 9. Bitsandbytes NF4 condition

In [ ]:
print("Loading HF model with bitsandbytes NF4 (4-bit)...")
hf_model_nf4 = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME, load_in_4bit=True, bnb_4bit_quant_type="nf4", device_map="auto"
)
hf_model_nf4.eval()
print("HF NF4 loaded.")

print("\nCaching NF4 activations...")
acts_nf4 = cache_hf_activations(hf_model_nf4, tokens_2d_cpu, LAYER, BATCH_SIZE)
ppl_nf4 = compute_hf_perplexity(hf_model_nf4, tokens_2d_cpu, BATCH_SIZE)

features_nf4 = sae_encode_batched(sae, acts_nf4, device)
corrs_nf4 = per_feature_pearson(hf_fp16_reference_features, features_nf4)
summary_nf4 = summarize(corrs_nf4, fp16_firing_rate)

print(f"\n=== bitsandbytes NF4 ===")
print(f"  ppl: {ppl_nf4:.3f} (Δ vs HF FP16: {(ppl_nf4/ppl_hf_fp16-1)*100:+.2f}%)")
print(f"  survived: {summary_nf4['survived_pct']:.2f}%")
print(f"  damaged: {summary_nf4['damaged_pct']:.2f}%")

torch.save(corrs_nf4, CKPT_2B("phase2b_corrs_BNB_NF4.pt"))

del hf_model_nf4, acts_nf4, features_nf4
gc.collect()
torch.cuda.empty_cache()

## 10. Phase 2B summary table

In [ ]:
# Assemble all Phase 2B conditions plus the matching RTN condition from Phase 2A for comparison
target_row_dict = target_row.to_dict()
target_row_dict["method"] = "simulated_rtn"
target_row_dict["param"] = int(target_row_dict["bits"])

phase2b_rows = [
    target_row_dict,  # the matched RTN condition from Phase 2A
    prune_row,
    {
        "condition": "BNB_INT8", "method": "real_bnb", "param": 8,
        "perplexity": ppl_int8,
        "ppl_delta_pct": (ppl_int8/ppl_hf_fp16 - 1) * 100,
        **summary_int8,
    },
    {
        "condition": "BNB_NF4", "method": "real_bnb", "param": 4,
        "perplexity": ppl_nf4,
        "ppl_delta_pct": (ppl_nf4/ppl_hf_fp16 - 1) * 100,
        **summary_nf4,
    },
]
phase2b_df = pd.DataFrame(phase2b_rows)
phase2b_df.to_csv(CKPT_2B("phase2b_summary.csv"), index=False)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)
print(phase2b_df.to_string(index=False))

## 11. Matched-perplexity comparison: quantization vs pruning

The headline analysis. Compute:
- **Jaccard overlap** of damaged features between RTN-INT6 and matched-sparsity pruning
- **Per-feature damage-score correlation** between the two methods

Low overlap = different feature classes damaged = strong Borobia differentiation.

In [ ]:
# Load the matched RTN condition's per-feature correlations from Phase 2A
corrs_rtn = torch.load(CKPT_2A(f"phase2a_L{LAYER}_corrs_{PRUNING_MATCH_TARGET}.pt"))

active_idx = torch.where(fp16_firing_rate > 0.001)[0]

# Damage labels (binary)
DAMAGE_THRESHOLD = 0.5
damaged_rtn = corrs_rtn[active_idx] < DAMAGE_THRESHOLD
damaged_prune = corrs_prune[active_idx] < DAMAGE_THRESHOLD

n_rtn = damaged_rtn.sum().item()
n_prune = damaged_prune.sum().item()
n_both = (damaged_rtn & damaged_prune).sum().item()
n_either = (damaged_rtn | damaged_prune).sum().item()

jaccard = n_both / max(n_either, 1)

print(f"=== Damaged-feature overlap at matched perplexity ===")
print(f"  RTN-INT6 ppl delta: {target_row_dict['ppl_delta_pct']:.2f}%")
print(f"  Pruning ppl delta:  {prune_row['ppl_delta_pct']:.2f}%")
print(f"  ")
print(f"  Features damaged by RTN only:    {n_rtn - n_both}")
print(f"  Features damaged by pruning only: {n_prune - n_both}")
print(f"  Features damaged by both:        {n_both}")
print(f"  Features damaged by either:      {n_either}")
print(f"  ")
print(f"  Jaccard overlap: {jaccard:.3f}")
print(f"    (1.0 = identical feature damage, 0.0 = completely disjoint)")
print(f"  ")

# Per-feature damage-score correlation
# Damage score = 1 - correlation, restricted to active features
damage_rtn_scores = 1 - corrs_rtn[active_idx].float()
damage_prune_scores = 1 - corrs_prune[active_idx].float()
pearson_r = torch.corrcoef(torch.stack([damage_rtn_scores, damage_prune_scores]))[0, 1].item()
spearman_r = pd.Series(damage_rtn_scores.numpy()).corr(
    pd.Series(damage_prune_scores.numpy()), method="spearman"
)
print(f"  Per-feature damage-score correlation:")
print(f"    Pearson:  {pearson_r:.3f}")
print(f"    Spearman: {spearman_r:.3f}")
print(f"    (high = methods damage features in similar order; low = methods target different features)")

# Save the comparison data
overlap_df = pd.DataFrame({
    "feature_id": active_idx.numpy(),
    "corr_RTN": corrs_rtn[active_idx].numpy(),
    "corr_PRUNE": corrs_prune[active_idx].numpy(),
    "damaged_RTN": damaged_rtn.numpy(),
    "damaged_PRUNE": damaged_prune.numpy(),
    "fp16_firing_rate": fp16_firing_rate[active_idx].numpy(),
})
overlap_df.to_csv(CKPT_2B("phase2b_matched_perplexity_overlap.csv"), index=False)
print(f"\nSaved per-feature comparison to {CKPT_2B('phase2b_matched_perplexity_overlap.csv')}")

## 12. Comparison plots: matched-perplexity quantization vs pruning

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: scatter of per-feature correlation, RTN vs Pruning
axes[0].scatter(corrs_rtn[active_idx].numpy(), corrs_prune[active_idx].numpy(),
                alpha=0.3, s=8)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y=x (identical damage)')
axes[0].axvline(0.5, color='red', alpha=0.3, linestyle=':')
axes[0].axhline(0.5, color='red', alpha=0.3, linestyle=':')
axes[0].set_xlabel(f"Correlation under {PRUNING_MATCH_TARGET}")
axes[0].set_ylabel(f"Correlation under matched-sparsity pruning")
axes[0].set_title(f"Per-feature damage: quantization vs pruning\n(matched perplexity, Pearson r={pearson_r:.2f})")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Panel 2: Venn-like bar chart of damaged categories
categories = ['RTN only', 'Both', 'Pruning only', 'Neither\n(survived)']
counts = [n_rtn - n_both, n_both, n_prune - n_both, len(active_idx) - n_either]
colors = ['steelblue', 'purple', 'crimson', 'lightgrey']
bars = axes[1].bar(categories, counts, color=colors, edgecolor='black')
for bar, c in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=11)
axes[1].set_ylabel("Number of active features")
axes[1].set_title(f"Damage overlap (matched perplexity)\nJaccard = {jaccard:.3f}")
axes[1].grid(True, alpha=0.3, axis='y')

# Panel 3: damage rate by firing-rate decile, comparison
firing_active = fp16_firing_rate[active_idx]
deciles = pd.qcut(firing_active.numpy(), q=10, labels=False, duplicates='drop')
decile_data = pd.DataFrame({
    'decile': deciles,
    'damaged_rtn': damaged_rtn.numpy(),
    'damaged_prune': damaged_prune.numpy(),
})
agg = decile_data.groupby('decile').agg({'damaged_rtn': 'mean', 'damaged_prune': 'mean'}).reset_index()
x = np.arange(len(agg))
w = 0.4
axes[2].bar(x - w/2, agg['damaged_rtn']*100, w, label=PRUNING_MATCH_TARGET, color='steelblue')
axes[2].bar(x + w/2, agg['damaged_prune']*100, w, label=f"Pruning {matched_sparsity:.2f}", color='crimson')
axes[2].set_xlabel("Firing-rate decile (low → high)")
axes[2].set_ylabel("% of features damaged in decile")
axes[2].set_title("Which feature rarities each method damages")
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(CKPT_2B("phase2b_matched_perplexity_comparison.png"), dpi=140, bbox_inches='tight')
plt.show()

## 13. Comparison plot: simulated RTN vs real bitsandbytes

In [ ]:
# Compare INT8 simulated RTN (from Phase 2A) to bnb INT8 (from Phase 2B)
corrs_rtn_int8 = torch.load(CKPT_2A(f"phase2a_L{LAYER}_corrs_RTN_INT8.pt"))
corrs_bnb_int8 = torch.load(CKPT_2B("phase2b_corrs_BNB_INT8.pt"))

# At matched method (both INT8), how do simulated and real compare?
rtn8_active = corrs_rtn_int8[active_idx]
bnb8_active = corrs_bnb_int8[active_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(rtn8_active.numpy(), bnb8_active.numpy(), alpha=0.3, s=8)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y=x')
axes[0].set_xlabel("Correlation under simulated RTN INT8")
axes[0].set_ylabel("Correlation under real bitsandbytes INT8")
axes[0].set_title("Simulated vs real INT8 quantization")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# All methods histogram overlay
axes[1].hist(rtn8_active.numpy(), bins=40, alpha=0.4, label="Simulated RTN INT8", color='steelblue')
axes[1].hist(bnb8_active.numpy(), bins=40, alpha=0.4, label="bitsandbytes INT8", color='orange')
corrs_bnb_nf4 = torch.load(CKPT_2B("phase2b_corrs_BNB_NF4.pt"))
axes[1].hist(corrs_bnb_nf4[active_idx].numpy(), bins=40, alpha=0.4, label="bitsandbytes NF4", color='crimson')
axes[1].set_xlabel("Per-feature Pearson correlation vs FP16")
axes[1].set_ylabel("Number of active features")
axes[1].set_title("Distribution of feature correlations by method")
axes[1].axvline(0.9, color='green', linestyle='--', alpha=0.6)
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.6)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CKPT_2B("phase2b_simulated_vs_real.png"), dpi=140, bbox_inches='tight')
plt.show()

## 14. What to send me when this finishes

1. **`phase2b_summary.csv`** — the main results table
2. **`phase2b_matched_perplexity_comparison.png`** — the 3-panel matched-perplexity plot (this is the most important Phase 2B figure)
3. **`phase2b_simulated_vs_real.png`** — the simulated-vs-real INT8 comparison
4. The printed Jaccard overlap, Pearson r, and Spearman ρ numbers from cell 11
5. The calibration result from cell 7 (HF vs TL feature correlation)

**What these will tell us:**

- If calibration is high (>0.9), the bitsandbytes results are directly comparable to Phase 2A. If it's lower, results carry an explicit methodological note.
- If Jaccard < 0.3 at matched perplexity, you have a strong claim that quantization and pruning damage *different* feature classes. This is the Borobia differentiation result. If Jaccard > 0.6, the two methods damage similar features and the differentiation is weaker.
- If simulated RTN INT8 and real bnb INT8 produce similar correlation distributions, your Phase 2A findings transfer to real deployment. If they differ substantially, that's a methodological note worth making explicit.

**What to do next:** Phase 3 (Gemma-2-2B) on vast.ai, or stop here and write the workshop paper. We'll decide based on what these numbers look like.